In [ ]:
!pip install ultralytics
!pip install boxmot

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 61.6 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=99d35a7481c5b7156420c0d028513a4753cc99f9b45bed8b72be73fc82b5e61e
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: regex
    Found 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip "/content/drive/MyDrive/datasets/av/progetto/test_set_last_version.zip"

Streaming output truncated to the last 5000 lines.
  inflating: test/SNMOT-194/img1/000313.jpg  
  inflating: test/SNMOT-194/img1/000314.jpg  
  inflating: test/SNMOT-194/img1/000315.jpg  
  inflating: test/SNMOT-194/img1/000316.jpg  
  inflating: test/SNMOT-194/img1/000317.jpg  
  inflating: test/SNMOT-194/img1/000318.jpg  
  inflating: test/SNMOT-194/img1/000319.jpg  
  inflating: test/SNMOT-194/img1/000320.jpg  
  inflating: test/SNMOT-194/img1/000321.jpg  
  inflating: test/SNMOT-194/img1/000322.jpg  
  inflating: test/SNMOT-194/img1/000323.jpg  
  inflating: test/SNMOT-194/img1/000324.jpg  
  inflating: test/SNMOT-194/img1/000325.jpg  
  inflating: test/SNMOT-194/img1/000326.jpg  
  inflating: test/SNMOT-194/img1/000327.jpg  
  inflating: test/SNMOT-194/img1/000328.jpg  
  inflating: test/SNMOT-194/img1/000329.jpg  
  inflating: test/SNMOT-194/img1/000330.jpg  
  inflating: test/SNMOT-194/img1/000331.jpg  
  inflating: test/SNMOT-194/img1/000332.jpg  
  inflating: test/SNMOT-194/i

In [ ]:
import torch

def print_vram(note=""):
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2
        max_allocated = torch.cuda.max_memory_allocated() / 1024**2

        print(
            f"[VRAM] {note} | "
            f"alloc={allocated:.2f} MB | "
            f"reserved={reserved:.2f} MB | "
            f"peak={max_allocated:.2f} MB"
        )

DETECTION + TRACKING CON VISUALIZZAZIONE E SALVATAGGIO FILE

In [ ]:
from google.colab.patches import cv2_imshow

In [ ]:
import cv2
import numpy as np

def draw_tracks(
    frame,
    tracks,
    model,
    colors,
    frame_count,
    mask=None  # maschera opzionale
):
    """
    Disegna bounding box, ID, classe e info sul frame.
    Se viene fornita una maschera, annerisce le parti del frame dove mask==0.

    frame: immagine BGR
    tracks: lista di tracce [x1, y1, x2, y2, track_id, conf, cls]
    model: modello che contiene i nomi delle classi
    colors: lista di colori per le tracce
    frame_count: numero del frame corrente
    mask: immagine mask 0/255
    """
    # Applica la maschera se fornita
    if mask is not None:
        if len(mask.shape) == 2:  # maschera singolo canale
            mask_3c = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        else:
            mask_3c = mask

        # Annerisce le parti dove mask==0
        frame = cv2.bitwise_and(frame, mask_3c)

    # Disegna i bounding box
    if len(tracks) > 0:
        for track in tracks:
            x1, y1, x2, y2 = map(int, track[:4])
            track_id = int(track[4])
            conf = track[5]
            cls = int(track[6])

            color = colors[track_id % len(colors)].tolist()

            # Bounding box
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)

            # Label
            class_name = model.names[cls]
            label = f"ID:{track_id} {class_name} {conf:.2f}"
            (w, h), _ = cv2.getTextSize(
                label, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 1
            )

            cv2.rectangle(frame, (x1, y1 - 20), (x1 + w, y1), color, -1)
            cv2.putText(
                frame,
                label,
                (x1, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                2
            )

    # Info frame
    info_text = f"Frame: {frame_count} | Tracce attive: {len(tracks)}"
    cv2.putText(
        frame,
        info_text,
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2
    )

    return frame


In [ ]:
def yolo_to_tracker(results):
    """
    Converte l'output YOLO in formato adatto al tracker.

    Output:
        np.ndarray di shape (N, 6)
        [x1, y1, x2, y2, confidence, class]
    """
    detections = []

    if len(results[0].boxes) > 0:
        boxes = results[0].boxes.xyxy.cpu().numpy()
        confidences = results[0].boxes.conf.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, conf, cls in zip(boxes, confidences, classes):
            detections.append([*box, conf, cls])

    return np.array(detections) if detections else np.empty((0, 6))


In [ ]:
import cv2
import numpy as np


def extract_field_mask_light(image_path):
    """
    Prende in input il percorso dell'immagine e restituisce
    la maschera binaria del campo da calcio.
    """
    # 1. Caricamento e conversione colore
    img = cv2.imread(image_path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    # 2. Range verde (campi da calcio)
    lower = np.array([35, 40, 40])
    upper = np.array([85, 255, 255])
    mask = cv2.inRange(hsv, lower, upper)

    # 3. Pulizia morfologica
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=3)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    # 4. Estrazione della componente più grande
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask)
    if num_labels > 1:
        largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
        field_mask = (labels == largest_label).astype(np.uint8) * 255
    else:
        field_mask = mask

    # 5. Dilatazione per espandere il campo e includere elementi interni
    kernel_large = np.ones((15, 15), np.uint8)
    field_mask = cv2.dilate(field_mask, kernel_large, iterations=2)

    # 6. Riempimento dei buchi interni (linee bianche, giocatori)
    contours, _ = cv2.findContours(field_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(field_mask)

    if contours:
        largest_contour = max(contours, key=cv2.contourArea)
        cv2.drawContours(
            filled_mask, [largest_contour], -1, 255, thickness=cv2.FILLED
        )

    # 7. Erosione per riportare il campo alle dimensioni originali
    filled_mask = cv2.erode(filled_mask, kernel_large, iterations=2)

    # 8. Dilatazione finale per allargare la maschera e includere i bordi del campo
    kernel_final = np.ones((21, 21), np.uint8)
    filled_mask = cv2.dilate(filled_mask, kernel_final, iterations=4)


    return filled_mask

In [ ]:
import cv2
import numpy as np

# funzione di visualizzazione dei campioni che sono fuori maschera

# questa funzione è stata usata per il debugging della pipeline che cancella le detections
# fuori maschera (Figura 9 nella documentazione), una volta stabilito che la pipeline funzionava 
# la funzione non è stata più usata

def visualize_result(image_path, field_mask, detections: np.ndarray = None):
    """
    Prende in input:
    - image_path: percorso dell'immagine
    - field_mask: maschera binaria del campo
    - detections: np.ndarray di shape (N, 6) -> [x1, y1, x2, y2, conf, cls]

    Visualizza:
    - immagine originale con bounding box
    - maschera
    - risultato finale (immagine mascherata) con bounding box
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Impossibile leggere l'immagine: {image_path}")

    result = cv2.bitwise_and(img, img, mask=field_mask)

    # Copie per il disegno
    img_vis = img.copy()
    result_vis = result.copy()

    # Disegno bounding box
    if detections is not None and len(detections) > 0:
        for det in detections:
            x1, y1, x2, y2, conf, cls = det.astype(int)

            # Colore bbox (verde)
            color = (0, 255, 0)
            thickness = 2

            # Bounding box
            cv2.rectangle(img_vis, (x1, y1), (x2, y2), color, thickness)
            cv2.rectangle(result_vis, (x1, y1), (x2, y2), color, thickness)

            # Etichetta
            label = f"cls:{int(cls)} conf:{conf:.2f}"
            cv2.putText(
                img_vis,
                label,
                (x1, max(y1 - 10, 0)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                1,
                cv2.LINE_AA,
            )

    #cv2.imshow("Originale + BBox", img_vis)
    #cv2.imshow("Maschera Piena (Senza Giocatori)", field_mask)
    #cv2.imshow("Risultato Finale + BBox", result_vis)
    cv2_imshow(result_vis)

    #cv2.waitKey(0)
    #cv2.destroyAllWindows()


In [ ]:
def filter_detections_by_mask(detections, mask, img_path=None, visualize_results=False):
    """
    Filtra le detections mantenendo solo quelle il cui centro della base è dentro il campo.

    detections: np.ndarray, shape (N,6) -> [x1, y1, x2, y2, conf, cls]
    mask: np.ndarray, binaria, 255 = campo, 0 = fuori campo

    return: np.ndarray filtrata
    """
    filtered = []
    deleted = []
    for det in detections:
        x1, y1, x2, y2, conf, cls = det
        cx = int((x1 + x2) / 2)
        cy = int(y2)

        # Controlla se le coordinate sono dentro l'immagine
        if 0 <= cx < mask.shape[1] and 0 <= cy < mask.shape[0]:
            if mask[cy, cx] == 255:
                filtered.append(det)
            else:
                #print("detection_deleted")
                deleted.append(det)

    if visualize_results and len(deleted) > 0 and img_path is not None:
        visualize_result(img_path, mask, deleted)

    if len(filtered) > 0:
        return np.array(filtered)
    else:
        return np.empty((0, 6))


In [ ]:
import csv
import os

def ordina_csv_per_track_id_txt(input_csv, output_txt):
    """
    Ordina un file CSV in base alla seconda colonna (track_id)
    e salva il risultato in un file TXT.
    """
    with open(input_csv, newline="") as f:
        reader = csv.reader(f)
        rows = list(reader)

    rows.sort(key=lambda x: int(x[1]))

    with open(output_txt, "w") as f:
        for row in rows:
            f.write(",".join(row) + "\n")

    os.remove(input_csv)

    print(f"File TXT ordinato salvato in {output_txt}")


In [ ]:
import os
import cv2
import csv
import numpy as np
from ultralytics import YOLO
from boxmot import ByteTrack

import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
cuda


In [ ]:
visualize_results = True

In [ ]:
import numpy as np

def percentage_masked_pixels(mask):
    """
    Calcola la percentuale di pixel oscurati (valore 0)
    in una maschera binaria.

    Parametri:
        mask (np.ndarray): immagine binaria (0 / 255 o 0 / 1)

    Ritorna:
        float: percentuale di pixel oscurati [0, 100]
    """
    if mask is None:
        raise ValueError("La maschera è None")

    total_pixels = mask.size
    if total_pixels == 0:
        return 0.0

    # Pixel oscurati = zero
    masked_pixels = np.count_nonzero(mask == 0)

    return (masked_pixels / total_pixels) * 100.0


behaviour

In [ ]:
import json

def load_rois_absolute(json_path, img_width, img_height):
    """
    Legge un JSON con ROI in coordinate relative e le converte in coordinate assolute.

    :param json_path: path al file JSON
    :param img_width: larghezza dell'immagine (pixel)
    :param img_height: altezza dell'immagine (pixel)
    :return: lista di tuple (x, y, width, height) in coordinate assolute
    """
    with open(json_path, "r") as f:
        rois = json.load(f)

    absolute_rois = []

    for roi_name, roi in rois.items():
        x_abs = roi["x"] * img_width
        y_abs = roi["y"] * img_height
        width_abs = roi["width"] * img_width
        height_abs = roi["height"] * img_height

        absolute_rois.append((x_abs, y_abs, width_abs, height_abs))

    return absolute_rois

def is_bbox_base_center_inside_roi(roi, x1_i, y1_i, bw_i, bh_i):
    """
    Verifica se il centro della base di un bounding box è dentro una ROI.

    :param roi: tupla (x_roi, y_roi, w_roi, h_roi) in coordinate assolute
    :param x1_i: top-left x del bounding box
    :param y1_i: top-left y del bounding box
    :param bw_i: larghezza del bounding box
    :param bh_i: altezza del bounding box
    :return: True se il centro della base è nella ROI, False altrimenti
    """
    x_roi, y_roi, w_roi, h_roi = roi

    # centro della base del bounding box
    cx = x1_i + bw_i / 2
    cy = y1_i + bh_i

    # limiti ROI
    inside_x = x_roi <= cx <= (x_roi + w_roi)
    inside_y = y_roi <= cy <= (y_roi + h_roi)

    return inside_x and inside_y

def append_rois_results_to_txt(rois_results, file_path):
    """
    Scrive i risultati delle ROI su file TXT, una riga per tupla, comma-separated.
    Se il file esiste, i nuovi dati vengono aggiunti in coda.

    :param rois_results: lista di tuple (frame_id, roi_index, count)
    :param file_path: path del file txt
    """
    with open(file_path, "a") as f:
        for result in rois_results:
            line = ",".join(str(v) for v in result)
            f.write(line + "\n")

In [ ]:
def tracking(model: YOLO, tracker, frames_folder, visualize_results=False, group_number=5, save_results=False, rois=None):

    video_writer = None
    fps = 30

    # colori per le bounding box
    np.random.seed(42)
    colors = np.random.randint(0, 255, size=(200, 3), dtype=np.uint8)

    frame_files = sorted([f for f in os.listdir(frames_folder) if f.endswith(('.jpg', '.png'))])

    numero_video = frames_folder.split("SNMOT-")[1].split("/")[0]
    numero_video = int(numero_video)

    #numero_video = int(frames_folder.split("/")[-2])

    # File di output
    output_csv = f"tracking_{numero_video:03d}_{group_number:02d}.csv"
    output_txt = f"tracking_{numero_video:03d}_{group_number:02d}.txt"
    output_behaviour = f"behavior_{numero_video:03d}_{group_number:02d}.txt"

    mask_light_count = 0
    mask_severe_count = 0

    # se non vogliamo elaborare tutti i frame di un video
    fps_limiter = 800
    fps_counter = 0

    with open(output_csv, mode="w", newline="") as f:
        writer = csv.writer(f)

        for file_name in frame_files:

            fps_counter += 1
            if fps_counter > fps_limiter:
              break

            roi_counts = [0] * len(rois)
            rois_results = []

            frame_path = os.path.join(frames_folder, file_name)
            frame = cv2.imread(frame_path)
            if frame is None:
                continue

            if video_writer is None:
                h, w, _ = frame.shape
                video_output_path = f"tracking_{numero_video:03d}_{group_number:02d}.mp4"
                fourcc = cv2.VideoWriter_fourcc(*"mp4v")
                video_writer = cv2.VideoWriter(video_output_path, fourcc, fps, (w, h))

            # frame_id: da "000025.jpg" a 25
            frame_id = int(os.path.splitext(file_name)[0])


            # 1: Detection
            results = model(
                frame,
                verbose=False,
                classes=[0],
                conf=0.25,
                imgsz=1024
            )

            # 2: Bounding boxes conversione a formato accettato da ByteTrack
            detections = yolo_to_tracker(results)

            # 2.5: Filtraggio detections
            mask_light = extract_field_mask_light(frame_path)
            if percentage_masked_pixels(mask_light) > 60:
                mask = None
            else:
                mask = mask_light
                detections = filter_detections_by_mask(detections, mask, frame_path, visualize_results)

            # 3: Update tracker
            tracks = tracker.update(detections, frame)

            # 4: Salvataggio tracce
            h_img, w_img, _ = frame.shape

            for track in tracks:
                x1, y1, x2, y2, track_id, conf, cls, _ = track

                # bounding box in formato top-left + width/height
                bw = x2 - x1
                bh = y2 - y1

                # arrotondamento
                x1_i = int(round(x1))
                y1_i = int(round(y1))
                bw_i = int(round(bw))
                bh_i = int(round(bh))

                # clamp ai bordi dell'immagine
                x1_i = max(0, min(x1_i, w_img - 1))
                y1_i = max(0, min(y1_i, h_img - 1))

                # evita box che escono
                bw_i = max(1, min(bw_i, w_img - x1_i))
                bh_i = max(1, min(bh_i, h_img - y1_i))

                writer.writerow([
                    frame_id,          # frame ID
                    int(track_id),     # track ID
                    x1_i,              # top-left x (int)
                    y1_i,              # top-left y (int)
                    bw_i,              # width (int)
                    bh_i,              # height (int)
                    round(float(conf), 3),       # confidence
                    int(cls),          # class
                    -1,                # MOT placeholder
                    -1                 # MOT placeholder
                ])

                # ROI check
                for i, roi in enumerate(rois):
                  is_bbox_inside_roi = is_bbox_base_center_inside_roi(roi, x1_i, y1_i, bw_i, bh_i)
                  if is_bbox_inside_roi:
                    roi_counts[i] += 1

            # ROI check
            for i, roi in enumerate(rois):
              rois_results.append((frame_id, i+1, roi_counts[i]))

            append_rois_results_to_txt(rois_results, output_behaviour)

            # 5: Visualizzazione
            frame = draw_tracks(
                frame=frame,
                tracks=tracks,
                model=model,
                colors=colors,
                frame_count=frame_id,
                mask=mask
            )

            # video
            video_writer.write(frame)

            if visualize_results:
              cv2_imshow(frame)
              if cv2.waitKey(1) & 0xFF == ord('q'):
                  break


    print(f"Elaborazione completata. Tracce salvate in {output_csv}")
    if visualize_results:
      cv2.destroyAllWindows()
    ordina_csv_per_track_id_txt(output_csv, output_txt)
    print(f"mask_light_count: {mask_light_count}")
    print(f"mask_severe_count: {mask_severe_count}")

    if video_writer is not None:
      video_writer.release()


In [ ]:
import os
from boxmot.trackers.bytetrack.basetrack import BaseTrack
from boxmot import BotSort

IMG_WIDTH = 1920
IMG_HEIGHT = 1080

frames_folder = "./test"

counter = 0

save_results = True
visualize_results = False
limit_video = 100

for entry in os.listdir(frames_folder):
    torch.cuda.reset_peak_memory_stats()  # resettare prima di ogni ciclo

    print_vram()

    BaseTrack._count = 0 # RESET ID GLOBALI

    model = None
    tracker = None

    model = YOLO('/content/best_YOLO12_v2.pt')
    model.to(device)


    # Inizializzazione del tracker BotSort con tutti i parametri
    tracker = BotSort(
        reid_weights='osnet_x0_25_msmt17.pt',  # Path ai pesi del modello ReID
        device=0,
        half=False,                             # Usa FP16 (half precision) per velocizzare

        # Parametri per il tracking
        track_high_thresh=0.6,        # Soglia di confidenza alta per le detection
        track_low_thresh=0.1,         # Soglia di confidenza bassa per le detection
        new_track_thresh=0.7,         # Soglia per creare nuove tracce
        track_buffer=30,              # Numero di frame per mantenere tracce perse
        match_thresh=0.8,             # Soglia di matching IoU per associazione

        # Parametri per il re-identification (ReID)
        proximity_thresh=0.5,         # Soglia di prossimità per ReID
        appearance_thresh=0.25,       # Soglia di similarità delle feature di appearance

        # Parametri del filtro di Kalman
        lambda_=0.98,                 # Fattore di smoothing per la stima della posizione

        # Parametri per la gestione delle tracce
        with_reid=True,               # Abilita il re-identification
        fuse_first_associate=False,   # Fonde first association con appearance features

        frame_rate=30,

        # Parametri per il CMC (Camera Motion Compensation)
        cmc_method='sof',   # Metodo per compensazione movimento camera

        # Parametri aggiuntivi
        per_class=False,              # Tracciamento separato per ogni classe
    )

    # Controlla se abbiamo già raggiunto le max iterazioni
    if counter >= limit_video:
        break

    print_vram()
    full_path = os.path.join(frames_folder, entry, "img1")

    roi_path = os.path.join(frames_folder, entry, "roi.json")
    if os.path.isfile(roi_path):
      rois_loaded = load_rois_absolute(roi_path, img_width=IMG_WIDTH, img_height=IMG_HEIGHT)

    if os.path.isdir(full_path):
        tracking(model, tracker, full_path, visualize_results, save_results=save_results, rois=rois_loaded)
        counter += 1

[VRAM]  | alloc=0.00 MB | reserved=0.00 MB | peak=0.00 MB


SUCCESS  | BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=30, match_thresh=0.8, proximity_thresh=0.5, appearance_thresh=0.25, cmc_method=sof, frame_rate=30, fuse_first_associate=False, with_reid=True, lambda_=0.98
INFO     | BoxMOT v16.0.5 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 3538] Downloading ReID weights from https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF → osnet_x0_25_msmt17.pt
Downloading...
From: https://drive.google.com/uc?id=1sSwXSUlj4_tHZequ_iZ8w_Jh0VaRQMqF
To: /content/osnet_x0_25_msmt17.pt
100%|██████████| 3.06M/3.06M [00:00<00:00, 300MB/s]
SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


[VRAM]  | alloc=78.14 MB | reserved=90.00 MB | peak=78.14 MB


SUCCESS  | BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=30, match_thresh=0.8, proximity_thresh=0.5, appearance_thresh=0.25, cmc_method=sof, frame_rate=30, fuse_first_associate=False, with_reid=True, lambda_=0.98
INFO     | BoxMOT v16.0.5 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 3538] Found existing ReID weights at osnet_x0_25_msmt17.pt; skipping download.


Elaborazione completata. Tracce salvate in tracking_190_05.csv
File TXT ordinato salvato in tracking_190_05.txt
mask_light_count: 0
mask_severe_count: 0
[VRAM]  | alloc=111.29 MB | reserved=448.00 MB | peak=111.29 MB


...

[VRAM]  | alloc=427.94 MB | reserved=708.00 MB | peak=427.94 MB


SUCCESS  | BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False, asso_func=iou, track_high_thresh=0.6, track_low_thresh=0.1, new_track_thresh=0.7, track_buffer=30, match_thresh=0.8, proximity_thresh=0.5, appearance_thresh=0.25, cmc_method=sof, frame_rate=30, fuse_first_associate=False, with_reid=True, lambda_=0.98
INFO     | BoxMOT v16.0.5 🚀 Python-3.12.12 torch-2.9.0+cu126
CUDA:0 (NVIDIA A100-SXM4-40GB, 40507MiB)
INFO     | osnet_x0_25_msmt17.pt
INFO     | [PID 3538] Found existing ReID weights at osnet_x0_25_msmt17.pt; skipping download.


Elaborazione completata. Tracce salvate in tracking_150_05.csv
File TXT ordinato salvato in tracking_150_05.txt
mask_light_count: 0
mask_severe_count: 0
[VRAM]  | alloc=428.09 MB | reserved=708.00 MB | peak=428.09 MB


SUCCESS  | Loaded pretrained weights from osnet_x0_25_msmt17.pt


[VRAM]  | alloc=112.02 MB | reserved=708.00 MB | peak=505.14 MB
Elaborazione completata. Tracce salvate in tracking_193_05.csv
File TXT ordinato salvato in tracking_193_05.txt
mask_light_count: 0
mask_severe_count: 0


In [ ]:
import os
from google.colab import files

def download_only_files():
    # Percorso della cartella principale di Colab
    path = '/content'

    # Lista tutti gli elementi nel percorso
    items = os.listdir(path)

    # Filtra tenendo solo i file (escludendo cartelle e file di sistema)
    files_to_download = [
        f for f in items
        if os.path.isfile(os.path.join(path, f)) and not f.startswith('.') and f.lower().endswith('.mp4')
    ]

    if not files_to_download:
        print("Nessun file trovato nella cartella /content.")
        return

    print(f"Trovati {len(files_to_download)} file. Inizio download...")

    for file_name in files_to_download:
        print(f"Download in corso: {file_name}")
        files.download(os.path.join(path, file_name))

download_only_files()

Trovati 49 file. Inizio download...
Download in corso: tracking_123_05.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download in corso: tracking_116_05.mp4


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download in corso: tracking_193_05.mp4


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>